In [1]:
import pandas as pd 
import numpy as np
import re

In [2]:
df = pd.read_csv("laptop_price.csv",encoding ="latin-1")
print("shape",df.shape)
print("\nMissing values",df.isnull().sum())
print("\nduplicate rows",df.duplicated().sum())

shape (1303, 13)

Missing values laptop_ID           0
Company             0
Product             0
TypeName            0
Inches              0
ScreenResolution    0
Cpu                 0
Ram                 0
Memory              0
Gpu                 0
OpSys               0
Weight              0
Price_euros         0
dtype: int64

duplicate rows 0


In [3]:
print("Dtypes before cleaning",df.dtypes)

Dtypes before cleaning laptop_ID             int64
Company              object
Product              object
TypeName             object
Inches              float64
ScreenResolution     object
Cpu                  object
Ram                  object
Memory               object
Gpu                  object
OpSys                object
Weight               object
Price_euros         float64
dtype: object


In [4]:
df['Ram'] = df['Ram'].str.replace('GB', '', regex=False).astype(int)
df['Weight'] = df['Weight'].str.replace('kg', '', regex=False).astype(float)

In [5]:
def parse_memory(mem_str):
    result = {'SSD': 0, 'HDD': 0, 'Flash_Storage': 0, 'Hybrid': 0}
    for part in mem_str.split('+'):
        part = part.strip()
        m = re.search(r'([\d.]+)\s*(TB|GB)', part)
        if not m:
            continue
        size = float(m.group(1))
        unit = m.group(2)
        if unit == 'TB':
            size *= 1000  # standardize everything to GB
        if 'SSD' in part:
            result['SSD'] += size
        elif 'HDD' in part:
            result['HDD'] += size
        elif 'Flash Storage' in part:
            result['Flash_Storage'] += size
        elif 'Hybrid' in part:
            result['Hybrid'] += size
    return result

parsed_memory = df['Memory'].apply(parse_memory).apply(pd.Series)
df = pd.concat([df, parsed_memory], axis=1)

In [7]:
df['Cpu_speed_GHz'] = df['Cpu'].str.extract(r'([\d.]+)GHz').astype(float)

def simplify_cpu(cpu):
    if 'Intel Core i7' in cpu: return 'Intel i7'
    elif 'Intel Core i5' in cpu: return 'Intel i5'
    elif 'Intel Core i3' in cpu: return 'Intel i3'
    elif 'Intel' in cpu: return 'Intel Other'
    elif 'AMD' in cpu: return 'AMD'
    else: return 'Other'

df['Cpu_brand'] = df['Cpu'].apply(simplify_cpu)

In [8]:
df['Gpu_brand'] = df['Gpu'].apply(lambda x: x.split()[0])

In [9]:
df['Touchscreen'] = df['ScreenResolution'].str.contains('Touchscreen').astype(int)
df['IPS'] = df['ScreenResolution'].str.contains('IPS').astype(int)

res = df['ScreenResolution'].str.extract(r'(\d+)x(\d+)')
df['X_res'] = res[0].astype(int)
df['Y_res'] = res[1].astype(int)
df['PPI'] = (((df['X_res']**2 + df['Y_res']**2) ** 0.5) / df['Inches']).round(2)

In [11]:
df_clean = df.drop(columns=[
    'laptop_ID', 'Product', 'Memory', 'Cpu', 'Gpu', 'ScreenResolution',
    'X_res', 'Y_res'
])

In [12]:
print("\nPrice skew (raw):", df_clean['Price_euros'].skew())
print("Price skew (log):", np.log1p(df_clean['Price_euros']).skew())


Price skew (raw): 1.5208655681688525
Price skew (log): -0.1723371919644384


In [13]:
numeric_cols = ['Ram','Weight','SSD','HDD','Flash_Storage','Hybrid',
                 'Cpu_speed_GHz','Touchscreen','IPS','PPI','Inches','Price_euros']
print("\nCorrelation with Price_euros:\n", df_clean[numeric_cols].corr()['Price_euros'].sort_values(ascending=False))

df_clean.to_csv('laptop_cleaned.csv', index=False)
print("\nSaved cleaned dataset -> laptop_cleaned.csv")
print("Final shape:", df_clean.shape)
print("Final columns:", list(df_clean.columns))


Correlation with Price_euros:
 Price_euros      1.000000
Ram              0.743007
SSD              0.670799
PPI              0.473506
Cpu_speed_GHz    0.430293
IPS              0.252208
Weight           0.210370
Touchscreen      0.191226
Inches           0.068197
Hybrid           0.007989
Flash_Storage   -0.040511
HDD             -0.096441
Name: Price_euros, dtype: float64

Saved cleaned dataset -> laptop_cleaned.csv
Final shape: (1303, 17)
Final columns: ['Company', 'TypeName', 'Inches', 'Ram', 'OpSys', 'Weight', 'Price_euros', 'SSD', 'HDD', 'Flash_Storage', 'Hybrid', 'Cpu_speed_GHz', 'Cpu_brand', 'Gpu_brand', 'Touchscreen', 'IPS', 'PPI']


# One hot encoding

In [14]:
import pandas as pd

df = pd.read_csv('laptop_cleaned.csv')

cat_cols = ['Company', 'TypeName', 'OpSys', 'Cpu_brand', 'Gpu_brand']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(df_encoded.shape)  # (1303, 51)

(1303, 51)


# Train test split

In [15]:
import numpy as np
from sklearn.model_selection import train_test_split

X = df_encoded.drop(columns=['Price_euros'])
y = np.log1p(df_encoded['Price_euros'])  # log transform since price is right-skewed

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model training

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 4 candidates to test
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)          # teach it using training data
    preds_log = model.predict(X_test)    # ask it to guess prices it hasn't seen
    
    preds_euros = np.expm1(preds_log)    # convert guess back to real euros
    actual_euros = np.expm1(y_test)      # convert true price back to real euros
    
    r2 = r2_score(y_test, preds_log)                              # how good the guesses are (0 to 1)
    rmse = np.sqrt(mean_squared_error(actual_euros, preds_euros))  # avg error in euros

    results.append({'Model': name, 'R2': round(r2,4), 'RMSE (euros)': round(rmse,2)})

print(pd.DataFrame(results).sort_values('R2', ascending=False))

               Model      R2  RMSE (euros)
3  Gradient Boosting  0.8780        302.33
2      Random Forest  0.8717        305.52
0  Linear Regression  0.8397        339.93
1      Decision Tree  0.8040        374.01


In [19]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [2, 3, 4],
}

grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV R2:", round(grid.best_score_, 4))

Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300}
Best CV R2: 0.8991


In [20]:
best_model = grid.best_estimator_
preds_log = best_model.predict(X_test)
preds_euros = np.expm1(preds_log)
actual_euros = np.expm1(y_test)

r2 = r2_score(y_test, preds_log)
rmse = np.sqrt(mean_squared_error(actual_euros, preds_euros))
print(f"Test R2: {r2:.4f}")
print(f"Test RMSE (euros): {rmse:.2f}")

Test R2: 0.8931
Test RMSE (euros): 296.07


In [21]:
import joblib

final_model = best_model
joblib.dump(final_model, 'laptop_price_model.pkl')
joblib.dump(list(X.columns), 'model_columns.pkl')
print("Model saved!")

Model saved!
